# Data Loading

In [48]:
import pandas as pd
import re
#import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [49]:
games_df = pd.read_csv('games.csv')
reviews_df = pd.read_csv('dataset.csv')
#reviews_df.head()

games_df.head()
reviews_df.head()

,app_id,app_name,review_text,review_score,review_votes
0,10,Counter-Strike,Ruined my life.,1,0
1,10,Counter-Strike,"This will be more of a ''my experience with this game'' type of review, because saying things like ''great gameplay'' will not suit something I've experienced with Counter-Strike. Here you go: I remember back in 2002 I was at a friend's house and he was playing a game. I didn't know the name of the game nor I had internet to find it. A few weeks passed by and another friend came over. He didn...",1,1
2,10,Counter-Strike,This game saved my virginity.,1,0
3,10,Counter-Strike,• Do you like original games? • Do you like games that don't lag? • Do you like games you can run on low end PC's? • Do you like games where you don't meet children who have slept with your mother? Then this is for you!,1,0
4,10,Counter-Strike,"Easy to learn, hard to master.",1,1


In [50]:
listed_genres = set()
for genres in games_df['Genres'].unique().tolist():
    for genre in str(genres).split(','):
        listed_genres.add(genre)


listed_tags = set()
for tags in games_df['Tags'].unique().tolist():
    for tag in str(tags).split(','):
        listed_tags.add(tag)

print(listed_genres)
print(len(listed_genres))

print(listed_tags)
print(len(listed_tags))

{'Utilities', 'nan', 'Audio Production', 'Web Publishing', 'Simulation', 'Software Training', 'Early Access', 'Tutorial', 'Free To Play', 'Adventure', 'Animation & Modeling', 'Photo Editing', 'Video Production', 'Short', 'Strategy', 'Violent', 'Massively Multiplayer', 'Racing', 'Education', 'Accounting', 'Sexual Content', 'Episodic', '360 Video', 'Action', 'Design & Illustration', 'Movie', 'Gore', 'Nudity', 'Casual', 'Game Development', 'Indie', 'RPG', 'Documentary', 'Sports'}
34
{'Runner', 'Atmospheric', 'Immersive', 'Hunting', 'Jump Scare', 'World War II', 'Remake', 'Board Game', 'Platformer', 'Souls-like', 'Golf', 'Dog', 'Tactical RPG', 'Old School', 'Underwater', 'Pinball', 'MOBA', 'Linear', 'Bowling', '360 Video', 'Co-op Campaign', 'Party', 'Economy', 'Mythology', 'Narration', 'Fast-Paced', 'Point & Click', 'Based On A Novel', 'Turn-Based Strategy', 'JRPG', 'Sokoban', 'Cinematic', "Shoot 'Em Up", 'Text-Based', 'Asymmetric VR', 'Crime', 'CRPG', 'Mystery Dungeon', 'PvE', 'Surreal', 

In [51]:

narrative_games = games_df[games_df['Tags'].str.contains('Story Rich') | games_df['Tags'].str.contains('Narrative') | games_df['Tags'].str.contains('Choices Matter')]

narrative_games.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
1108640,OMNIMUS,"Sep 25, 2019",0 - 20000,0,0,0.99,0,1,"Omnimus is an interactive visual novel in an unusual setting (a mix of science fiction and fantasy) that allows players to directly influence the development of its plot. Most states did not survive the third world war. The borders between them were erased, and a unified form of government was established on the planet - corporatocracy, which put power over people in the hands of cold-blooded ...","['English', 'Russian']",[],NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1108640/header.jpg?t=1754321404,https://vk.com/badnovells,NaN,letodigest@gmail.com,True,True,True,0,NaN,0,47,6,NaN,0,0,NaN,0,0,0,0,Angry games,Total,"Single-player,Family Sharing","Adventure,Indie","Adventure,Indie,Visual Novel,Interactive Fiction,Sci-fi,Anime,Singleplayer,Story Rich,Replay Value,Choices Matter,Multiple Endings","https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1108640/ss_bad5213815e94879830abb3dac22b1198de5ff29.1920x1080.jpg?t=1754321404,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1108640/ss_090d0efea0c0a0d3029956b52eb660f3d60bcce2.1920x1080.jpg?t=1754321404,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1108640/ss_de3700f45e6e2c06c73b91eb14...",NaN
1514380,COGEN: Sword of Rewind / COGEN: 大鳥こはくと刻の剣,"Jan 26, 2022",20000 - 50000,0,0,12.49,0,3,"■Defy Death by Rewinding Time! Any hit you take is a death sentence in this side-scrolling 2D action game. But fear not! You may lack the HP to weather a beating, but you've got the Ouroboros System on your side. This futuristic technology kicks in the instant death embraces you and allows you to rewind up to 3 seconds of time. Slip up and wish you could take it back? Want to play around and e...","['English', 'Simplified Chinese', 'Traditional Chinese', 'Japanese', 'Korean', 'Spanish - Spain', 'Portuguese - Brazil']","['Japanese', 'Portuguese - Brazil']",NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1514380/header.jpg?t=1700552737,https://cogen.city/,http://www.gemdrops.co.jp/,NaN,True,False,False,0,NaN,0,75,34,NaN,39,111,NaN,0,0,0,0,"Gemdrops, Inc.","Gemdrops, Inc.","Single-player,Steam Achievements,Full controller support,Steam Cloud,Family Sharing",Action,"Action-Adventure,Pixel Graphics,Time Manipulation,Side Scroller,2D,Female Protagonist,Action,Sci-fi,2D Platformer,Swordplay,Score Attack,Controller,1990's,Cute,Dynamic Narration,Story Rich,Tutorial,Futuristic,Singleplayer,Adventure","https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1514380/ss_b1135fbb0681a5695c34731bc6abab3e9758f55e.1920x1080.jpg?t=1700552737,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1514380/ss_0f68453b21f090c5bbe7c245fafb683173fa32e6.1920x1080.jpg?t=1700552737,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1514380/ss_4a4e77b1a79a73d82cea17b84f...",NaN
1479370,Pixel Game Maker Series Werewolf Princess Kaguya,"Dec 20, 2020",0 - 20000,0,0,9.99,0,0,"A stylish &amp; exciting action game! Journey through medieval Japan, rendered in beautiful pixel art, to reach the moon and defeat Mother Kaguya! Story: Mother Kaguya, Queen of the Moon, is planning to take over the world. Her daughter, Princess Kaguya, has been cursed to become a werewolf. Even so, she will stop at nothing to end her mother's evil plan. An epic battle between parent and chil...","['English', 'Simplified Chinese', 'Traditional Chinese', 'Japanese']",[],NaN,https://shar

In [52]:
narrative_game_names = narrative_games["AppID"].unique().tolist()

filtered_reviews = reviews_df[reviews_df["app_name"].isin(narrative_game_names)]

print(len(filtered_reviews))
print(len(filtered_reviews["app_name"].unique().tolist()))

filtered_reviews.head()


1236465
1101


,app_id,app_name,review_text,review_score,review_votes
46177,104600,Portal 2 - The Final Hours,"First of all, this is no game but a virtual interactive book, and a very good one as a matter of fact. To everyone who is interested in the behind the scenes of Portal 2, and how a development of a game works at Valve, you're in for one hell of a ride. I absolutely recommend it, it shows the everyday routine of an employee at Valve, some very enlightening points of view on their known delays a...",1,1
46178,104600,Portal 2 - The Final Hours,"The Final Hours is not a game (though can't you imagine one - set perhaps during the Seven Hour War, trying to escape as the facility collapses and the Combine invade the test chambers?), it's a book. It covers how Portal was created, the complex gestation of the sequel, and the personalities involved at every step. The book is interesting, revealing a lot of behind the scenes info. You learn ...",1,1
46179,104600,Portal 2 - The Final Hours,Get this if you are a geek about how games are made!,1,1
46180,104600,Portal 2 - The Final Hours,"IT'S NOT A GAME! But it's a very interesting reading if you are interested in the creative process behind Portal and how everything in Valve works. Some things i already knew, but i learned many after reading it. The interactive features also turns it more easy to read. I'll recommend it only if you are a Portal/Valve fan and you want to know more about how it was made, but it worth the low pr...",1,1
46181,104600,Portal 2 - The Final Hours,"Everything you would want to know about Portal is right here in this interactive book. If you're a Portal fanatic, this is a must-buy. This interactive book contains behind-the-scenes information, videos, polls, and more. It's all you could have ever wanted.",1,1


## Data Cleaning

In [53]:
#filters for ascii characters
filter_ascii = filtered_reviews["review_text"].replace(to_replace=r'[^\x00-\x7f]', value='', regex=True)

#removes new lines
remove_newline = filter_ascii.replace(to_replace=r'\d', value='', regex=True)

#removes certain characters used in reviews
remove_otherchars = remove_newline.replace(to_replace=r'[,()*/"1234567890+|]', value='', regex=True)

#removes punctuations
replace_punctuations = remove_otherchars.replace(to_replace=r'[.!?]+', value = ' | ', regex=True)

#remove dashes not used to hyphenate words
remove_nonhyphen_dashes = replace_punctuations.replace(to_replace=r'- | -', value=' ', regex=True)

#remove extra whitespace
remove_extra_whitespace = remove_nonhyphen_dashes.replace(to_replace=r' +', value=' ', regex=True)

# assign cleaned entries back to the data frame
new = remove_extra_whitespace
filtered_reviews['review_text'] = new

# apply lowercasing
filtered_reviews['review_text'] = filtered_reviews['review_text'].str.lower()
filtered_reviews

,app_id,app_name,review_text,review_score,review_votes
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual interactive book and a very good one as a matter of fact | to everyone who is interested in the behind the scenes of portal and how a development of a game works at valve you're in for one hell of a ride | i absolutely recommend it it shows the everyday routine of an employee at valve some very enlightening points of view on their known delays and som...,1,1
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you imagine one set perhaps during the seven hour war trying to escape as the facility collapses and the combine invade the test chambers | it's a book | it covers how portal was created the complex gestation of the sequel and the personalities involved at every step | the book is interesting revealing a lot of behind the scenes info | you learn what ...,1,1
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are made |,1,1
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting reading if you are interested in the creative process behind portal and how everything in valve works | some things i already knew but i learned many after reading it | the interactive features also turns it more easy to read | i'll recommend it only if you are a portalvalve fan and you want to know more about how it was made but it worth the low p...,1,1
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal is right here in this interactive book | if you're a portal fanatic this is a must-buy | this interactive book contains behind-the-scenes information videos polls and more | it's all you could have ever wanted |,1,1
...,...,...,...,...,...
6404109,9880,Champions Online,the gane starts to get boreing when you reatch level because you know every thats going to happine like all the oher classes go throught the same thing nonthing changes you can do diff storys but still the same i played a wile back but now its just to boreing all the updates are really plane they and new classes really they just changeing a diff one up,-1,0
6404110,9880,Champions Online,best character creator i've seen | edit: they removed items to costumize your character that were in the game before for free just so they can get extra cash |,-1,0
6404111,9880,Champions Online,region banned | why did u show me this game to us | |,-1,0
6404112,9880,Champions Online,why the,-1,0


In [54]:
## additional cleaning based on Kaggle User Ali Marjani, PhD
## via https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

print("before removing duplicates: " + str(filtered_reviews.shape))

# drop duplicates
## NOTE: only this actually decreased stuff lmao
filtered_reviews.drop_duplicates(['review_text', 'review_score'], inplace = True)
print("after removing duplicates: " + str(filtered_reviews.shape))


before removing duplicates: (1236465, 5)
after removing duplicates: (1017234, 5)


In [55]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

word_tokens = [ (str(review)).split() for review in filtered_reviews['review_text']]

stop_words = set(stopwords.words('english'))
stop_words.add('us')

lemmatized_reviews = []
for tokens in word_tokens:
    lemmatized_reviews.append([lemmatizer.lemmatize(token) for token in tokens if token not in stop_words or token=='not'])

lemmatized_reviews = [" ".join(review) for review in lemmatized_reviews]

filtered_reviews["lemmatized_reviews"] = lemmatized_reviews
filtered_reviews

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual interactive book and a very good one as a matter of fact | to everyone who is interested in the behind the scenes of portal and how a development of a game works at valve you're in for one hell of a ride | i absolutely recommend it it shows the everyday routine of an employee at valve some very enlightening points of view on their known delays and som...,1,1,first game virtual interactive book good one matter fact | everyone interested behind scene portal development game work valve one hell ride | absolutely recommend show everyday routine employee valve enlightening point view known delay video plus slideshows keep interested reading | read calmly attention take bit less hour go assure absolutely worth |
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you imagine one set perhaps during the seven hour war trying to escape as the facility collapses and the combine invade the test chambers | it's a book | it covers how portal was created the complex gestation of the sequel and the personalities involved at every step | the book is interesting revealing a lot of behind the scenes info | you learn what ...,1,1,final hour not game though can't imagine one set perhaps seven hour war trying escape facility collapse combine invade test chamber | book | cover portal created complex gestation sequel personality involved every step | book interesting revealing lot behind scene info | learn complicated business creating computer game especially valve one something stress involved | even company like valve c...
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are made |,1,1,get geek game made |
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting reading if you are interested in the creative process behind portal and how everything in valve works | some things i already knew but i learned many after reading it | the interactive features also turns it more easy to read | i'll recommend it only if you are a portalvalve fan and you want to know more about how it was made but it worth the low p...,1,1,not game | interesting reading interested creative process behind portal everything valve work | thing already knew learned many reading | interactive feature also turn easy read | recommend portalvalve fan want know made worth low price | fan buy
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal is right here in this interactive book | if you're a portal fanatic this is a must-buy | this interactive book contains behind-the-scenes information videos polls and more | it's all you could have ever wanted |,1,1,everything would want know portal right interactive book | portal fanatic must-buy | interactive book contains behind-the-scenes information video poll | could ever wanted |
...,...,...,...,...,...,...
6404107,9880,Champions Online,i paid for a month subscription to check out as much of the full game as i could | i spent nearly two hours building my hero making sure not to use pieces that cost zen there is a large pop up in the right corner that tells you that you're using a piece that cost zen so it's obvious when you are | i get in game and my entire hero is changed | everything but base colors and the chest emblem | i...,-1,1,paid month subscription check much full game could | spent nearly two hour building hero making sure not use piece cost zen large pop right corner tell using piece cost zen obvious | get game entire hero changed | everything base color chest emblem | log delete start piece used cost zen reason can't make hero want even though paying monthly subscription fee | waiting support give refund subscr...
6404108,9880,Champions Online,my absolute favorite superhero games even though the graphics is dated | i had a lifetime membership t

# Exploratory Data Analysis

In [56]:
#inserting data from 1 dataset to the next
estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
to_append = [False, False, False]
for i in range (len(filtered_reviews)):
    game = filtered_reviews.iloc[i,1] #sometimes its 1, sometimes its 2,,, idk why 
    if game != last_game:
        estimate = narrative_games.loc[narrative_games["AppID"]==game, "Release date"].iloc[0]
        #for some reason realease date is the header for estimated owners dont ask me
        game_tags = narrative_games.loc[narrative_games["AppID"]==game, "Tags"].iloc[0]
        to_append[0] = "Story Rich" in game_tags
        to_append[1] = "Narrative" in game_tags
        to_append[2] = "Choices Matter" in game_tags
        #same here,, required age = price
        price = narrative_games.loc[narrative_games["AppID"]==game, "Required age"].iloc[0]

    estimated_owners.append(estimate)
    story_rich.append(to_append[0])
    narrative.append(to_append[1])
    choices_matter.append(to_append[2])
    prices.append(price)
    last_game = game    


filtered_reviews["estimated_owners"] = estimated_owners 
filtered_reviews["story_rich"] = story_rich
filtered_reviews["narrative"] = narrative 
filtered_reviews["choices_matter"] = choices_matter 
filtered_reviews["price"] = prices


filtered_reviews.head()

KeyboardInterrupt: 

### enable gpu

In [ ]:
import torch

# Check if a CUDA-capable GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This line checks if cuda GPU is available
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.14.1+cu126
True
NVIDIA GeForce RTX 3060 Laptop GPU


# Sentiment Analysis

In [ ]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(lemmatized_reviews)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    com_sent.append(doc._.sentiment['compound'])
    pos_sent.append(doc._.sentiment['pos'])
    neg_sent.append(doc._.sentiment['neg'])
    neu_sent.append(doc._.sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(filtered_reviews)))
filtered_reviews['com_sent'] = com_sent
filtered_reviews['pos_sent'] = pos_sent
filtered_reviews['neg_sent'] = neg_sent
filtered_reviews['neu_sent'] = neu_sent

filtered_reviews.head()

35000/1236465


KeyboardInterrupt: 

# Topic Modeling

## Auto Topic Modeling with Transformers
Based on the methodology of Kaggle User Ali Marjani, PhD in the following notebook: https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

In [ ]:
import random as rn
import numpy as np 
import pandas as pd 
import os
import collections
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

pd.set_option("display.max_rows", 600)
pd.set_option("display.max_columns", 500)
pd.set_option("max_colwidth", 400)

In [159]:
SAMPLE_N = 2000

willrecco_df = filtered_reviews[filtered_reviews.review_score==1]
willrecco_reviews = (willrecco_df["review_text"].sample(n=SAMPLE_N, random_state = 1234)).tolist()
print(len(willrecco_reviews))

notrecco_df = filtered_reviews[filtered_reviews.review_score==-1]
notrecco_reviews = (filtered_reviews["review_text"].sample(n=SAMPLE_N, random_state = 1234)).tolist()
print(len(notrecco_reviews))

### WANT TO DO:
### - split into sentences (DONE)
### - filter out sentences with <5 words (DONE)
### - apply stopword removal + lemmatization(DONE??)

willrecco_sentences = []
for i, review in enumerate(willrecco_reviews):
    for sentence in (str(review)).split(" | "):
        if len(sentence.split()) >= 5:
            removed_stopwords = [lemmatizer.lemmatize(word) for word in sentence.split() if word.isalnum() and word not in stop_words]
            willrecco_sentences.append((" ").join(removed_stopwords))

notrecco_sentences = []
for i, review in enumerate(notrecco_reviews):
    for sentence in (str(review)).split(" | "):
        if len(sentence.split()) >= 5:
            removed_stopwords = [lemmatizer.lemmatize(word) for word in sentence.split() if word.isalnum() and word not in stop_words]
            notrecco_sentences.append((" ").join(removed_stopwords))

print(willrecco_sentences[:10])
print(notrecco_sentences[:10])

willrecco_reviews = willrecco_sentences
notrecco_reviews = notrecco_sentences

2000
2000
['game actually made want take video game art seriously', 'awesome game like previous two', 'might feel atmosphere first two game nice storyline fun vibrant city run around kill enemy', 'expect much dlc played ba ep found motivation play second episode yet main story good enough make disappointment dlc season pas bought game first came', 'seriously buy game sale worth every pennyeuro', 'gamer dont collection', 'fix fps', 'say buy multiplayer addicting really never get boring', 'one game rush enjoy', 'even time feel bit may seem bit repetitve']
['original sin provides hunger seeking', 'give feeling adventure felt kid looking rpgs alike', 'hard first pt easy rewarding fun', 'buy dont like first comeback take slow', 'source hunter take hisher time', 'nice game lack freedom world gta wannabe style', 'hyperactive combat feel like injecting coffee straight brain', 'tongue cheek plot parodying crackpot conspiracy', 'deliciously cringeworthy main character edge cut anything anyone le

In [143]:
from sentence_transformers import SentenceTransformer
import tensorflow as tf

In [144]:
# To limit allocating the whole GPU memory by Pytorch run this cell. 
# Source: https://www.tensorflow.org/guide/gpu

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

In [145]:
import umap # dimensionality reduction
import hdbscan # clustering
from functools import partial

# To perform the Bayesian Optimization for searching the optimum hyperparameters, 
# we use hyperopt package:
from hyperopt import hp
from hyperopt import fmin, tpe, hp, STATUS_OK, space_eval, Trials

### Function Definitions
embed()

generate_clusters(), score_clusters(), objective(), bayesian_search()

compute_IDF(), get_group(), most_common(), extract_labels(), apply_and_summarize_labels()

In [ ]:
def embed(model, model_type, sentences):
    if model_type == 'use':
        embeddings = model(sentences)
    elif model_type == 'sentence transformer':
        embeddings = model.encode(sentences)
    
    return embeddings

In [146]:
def generate_clusters(message_embeddings,
                      n_neighbors,
                      n_components, 
                      min_cluster_size,
                      min_samples = None,
                      random_state = None):
    """
    Returns HDBSCAN objects after first performing dimensionality reduction using UMAP
    
    Arguments:
        message_embeddings: embeddings to use
        n_neighbors: int, UMAP hyperparameter n_neighbors
        n_components: int, UMAP hyperparameter n_components
        min_cluster_size: int, HDBSCAN hyperparameter min_cluster_size
        min_samples: int, HDBSCAN hyperparameter min_samples
        random_state: int, random seed
        
    Returns:
        clusters: HDBSCAN object of clusters
    """
    
    umap_embeddings = (umap.UMAP(n_neighbors = n_neighbors, 
                                n_components = n_components, 
                                metric = 'cosine', 
                                random_state=random_state,
                                compatibility_layout=False
                                )
                            .fit_transform(message_embeddings))

    clusters = hdbscan.HDBSCAN(min_cluster_size = min_cluster_size, 
                               min_samples = min_samples,
                               metric='euclidean', 
                               gen_min_span_tree=True,
                               cluster_selection_method='eom').fit(umap_embeddings)
    
    return clusters

def score_clusters(clusters, prob_threshold = 0.05):
    """
    Returns the label count and cost of a given clustering

    Arguments:
        clusters: HDBSCAN clustering object
        prob_threshold: float, probability threshold to use for deciding
                        what cluster labels are considered low confidence

    Returns:
        label_count: int, number of unique cluster labels, including noise
        cost: float, fraction of data points whose cluster assignment has
              a probability below cutoff threshold
    """
    
    cluster_labels = clusters.labels_
    label_count = len(np.unique(cluster_labels))
    total_num = len(clusters.labels_)
    cost = (np.count_nonzero(clusters.probabilities_ < prob_threshold)/total_num)
    
    return label_count, cost

def objective(params, embeddings, label_lower, label_upper):
    """
    Objective function for hyperopt to minimize

    Arguments:
        params: dict, contains keys for 'n_neighbors', 'n_components',
               'min_cluster_size', 'random_state' and
               their values to use for evaluation
        embeddings: embeddings to use
        label_lower: int, lower end of range of number of expected clusters
        label_upper: int, upper end of range of number of expected clusters

    Returns:
        loss: cost function result incorporating penalties for falling
              outside desired range for number of clusters
        label_count: int, number of unique cluster labels, including noise
        status: string, hypoeropt status

        """
    
    clusters = generate_clusters(embeddings, 
                                 n_neighbors = params['n_neighbors'], 
                                 n_components = params['n_components'], 
                                 min_cluster_size = params['min_cluster_size'],
                                 random_state = params['random_state'])
    
    label_count, cost = score_clusters(clusters, prob_threshold = 0.05) # 0.05
    
    #15% penalty on the cost function if outside the desired range of groups
    if (label_count < label_lower) | (label_count > label_upper):
        penalty = 1.0 #0.5 
    else:
        penalty = 0
    
    loss = cost + penalty
    
    return {'loss': loss, 'label_count': label_count, 'status': STATUS_OK}

def bayesian_search(embeddings, space, label_lower, label_upper, max_evals=100):
    """
    Perform bayesian search on hyperparameter space using hyperopt

    Arguments:
        embeddings: embeddings to use
        space: dict, contains keys for 'n_neighbors', 'n_components',
               'min_cluster_size', and 'random_state' and
               values that use built-in hyperopt functions to define
               search spaces for each
        label_lower: int, lower end of range of number of expected clusters
        label_upper: int, upper end of range of number of expected clusters
        max_evals: int, maximum number of parameter combinations to try

    Saves the following to instance variables:
        best_params: dict, contains keys for 'n_neighbors', 'n_components',
               'min_cluster_size', 'min_samples', and 'random_state' and
               values associated with lowest cost scenario tested
        best_clusters: HDBSCAN object associated with lowest cost scenario
                       tested
        trials: hyperopt trials object for search

        """
    
    trials = Trials()
    fmin_objective = partial(objective, 
                             embeddings=embeddings, 
                             label_lower=label_lower,
                             label_upper=label_upper)
    
    best = fmin(fmin_objective, 
                space = space, 
                algo=tpe.suggest,
                max_evals=max_evals, 
                trials=trials)

    best_params = space_eval(space, best)
    print ('best:')
    print (best_params)
    print (f"label count: {trials.best_trial['result']['label_count']}")
    
    best_clusters = generate_clusters(embeddings, 
                                      n_neighbors = best_params['n_neighbors'], 
                                      n_components = best_params['n_components'], 
                                      min_cluster_size = best_params['min_cluster_size'],
                                      random_state = best_params['random_state'])
    
    return best_params, best_clusters, trials

In [147]:
def compute_IDF(documents):
    word_count = Counter()
    for doc in documents:
        if 'drops(players' in doc:
            print(doc)
            print(doc.split())
        words_set = set(doc.split())
        word_count.update(words_set)
    total = sum(word_count.values())
    return {k: round((np.log2(total / v)))  for k, v in word_count.items()} # log2 is the best choice for our work (feel free)
                                                                            # to try different functions.

def get_group(df, category_col, category):
    """
    Returns documents of a single category
    
    Arguments:
        df: pandas dataframe of documents
        category_col: str, column name corresponding to categories or clusters
        category: int, cluster number to return
    Returns:
        single_category: pandas dataframe with documents from a single category
    """
    
    single_category = df[df[category_col]==category].reset_index(drop=True)

    return single_category 

def most_common(lst, n_words):
    """
    Get most common words in a list of words
    
    Arguments:
        lst: list, each element is a word
        n_words: number of top common words to return
    
    Returns:
        counter.most_common(n_words): counter object of n most common words
    """
    counter=collections.Counter(lst)
    
    for k in list(counter): 
        if counter[k] ==1: # if appears only once, ignore it.
            pass 
        else:
            counter[k] *= word_IDF[k] # if the word appears more than once in the entire cluser, 
                                      # repeat that word "IDF" times in our bag. If a word is 
                                      # low-frequent word it has a high IDF values, so, with this
                                      # technique we give more chance to this word to show up in 
                                      # the list of most common words
        
    return counter.most_common(n_words)

def extract_labels(category_docs, print_word_counts=False):
    """
    Extract labels from documents in the same cluster by concatenating
    most common verbs, ojects, and nouns

    Argument:
        category_docs: list of documents, all from the same category or
                       clustering
        print_word_counts: bool, True will print word counts of each type in this category

    Returns:
        label: str, group label derived from concatentating most common
               verb, object, and two most common nouns

    """

    verbs = []
    dobjs = []
    nouns = []
    adjs = []
    
    verb = ''
    dobj = ''
    noun1 = ''
    noun2 = ''

    # for each document, append verbs, dobs, nouns, and adjectives to 
    # running lists for whole cluster
    for i in range(len(category_docs)):
        doc = nlp(category_docs[i])
        for token in doc:
            if (token.is_stop==False) and (len(str(token).strip()) > 0): 
                # ignore if it is a stop word or the length of stripped token is less than 1!
                if token.pos_ == 'VERB':
                    verbs.extend([token.lemma_.lower()]) 

                elif token.dep_=='dobj':
                    dobjs.extend([token.lemma_.lower()]) 

                elif token.pos_=='NOUN':
                    nouns.extend([token.lemma_.lower()]) 
                    
                elif token.pos_=='ADJ':
                    adjs.extend([token.lemma_.lower()])

    # for printing out for inspection purposes
    if print_word_counts:
        for word_lst in [verbs, dobjs, nouns, adjs]:
            counter=collections.Counter(word_lst)
            print(counter)
    
    # take most common words of each form
    if len(verbs) > 0:
        verb = most_common(verbs, 1)[0][0]
    
    if len(dobjs) > 0:
        dobj = most_common(dobjs, 1)[0][0]
    
    if len(nouns) > 0:
        noun1 = most_common(nouns, 1)[0][0]
    
    if len(set(nouns)) > 1:
        noun2 = most_common(nouns, 2)[1][0]
    
    # concatenate the most common verb-dobj-noun1-noun2 (if they exist)
    label_words = [verb, dobj]
    
    for word in [noun1, noun2]:
        if word not in label_words:
            label_words.append(word)
    
    if '' in label_words:
        label_words.remove('')
    
    label = '_'.join(label_words)
    
    return label

def apply_and_summarize_labels(df, category_col):
    """
    Assign groups to original documents and provide group counts

    Arguments:
        df: pandas dataframe of original documents of interest to
            cluster
        category_col: str, column name corresponding to categories or clusters

    Returns:
        summary_df: pandas dataframe with model cluster assignment, number
                    of documents in each cluster and derived labels
    """
    
    numerical_labels = df[category_col].unique()
    
    # create dictionary of the numerical category to the generated label
    label_dict = {}
    for label in numerical_labels:
        current_category = list(get_group(df, category_col, label)['text'])
        label_dict[label] = extract_labels(current_category)
        
    # create summary dataframe of numerical labels and counts
    summary_df = (df.groupby(category_col)['text'].count()
                    .reset_index()
                    .rename(columns={'text':'count'})
                    .sort_values('count', ascending=False))
    
    # apply generated labels
    summary_df['label'] = summary_df.apply(lambda x: label_dict[x[category_col]], axis = 1)
    
    return summary_df

In [ ]:
model_st1 = SentenceTransformer('all-mpnet-base-v2')

hspace = {
    "n_neighbors": hp.choice('n_neighbors', range(3,32)),
    "n_components": hp.choice('n_components', range(3,32)),
    "min_cluster_size": hp.choice('min_cluster_size', range(2,32)),
    "random_state": 42
}

label_lower = 10
label_upper = 100
max_evals = 25 # change it to 50 or 100 for extra steps as you wish.

willrecco_embeddings = embed(model_st1, 'sentence transformer', willrecco_reviews)

In [ ]:
def perform_topic_modeling(inputs: list[str]):
    embeddings = embed(model_st1, 'sentence transformer', inputs)

    # %%timet
    best_params_use, best_clusters_use, trials_use = bayesian_search(embeddings_st1, 
                                                                    space=hspace, 
                                                                    label_lower=label_lower, 
                                                                    label_upper=label_upper, 
                                                                    max_evals=max_evals)

    data_clustered = pd.DataFrame(data = list(zip(willrecco_reviews,best_clusters_use.labels_)),
                             columns = ['text', 'label_st1'])
    print("data_clustered.head()")
    print(data_clustered.head())

    sent_with_word_lemma = []
    for review in willrecco_reviews:
        doc = nlp(review)
        sent_temp = ""
        this_one = False
        for token in doc:
            if (token.pos_ in ['VERB', 'NOUN', 'ADJ']) or (token.dep_=='dobj'):
                sent_temp += token.lemma_.lower() + " "
        sent_with_word_lemma.append(sent_temp)

    word_IDF = compute_IDF(sent_with_word_lemma)
    cluster_summary = apply_and_summarize_labels(data_clustered, 'label_st1')
    print(cluster_summary.head())

perform_topic_modeling(willrecco_reviews)

 80%|████████  | 20/25 [03:37<01:00, 12.13s/trial, best loss: 0.0027436478587617797]

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a Num

 84%|████████▍ | 21/25 [03:48<00:48, 12.05s/trial, best loss: 0.0027436478587617797]

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a Num

 88%|████████▊ | 22/25 [04:02<00:37, 12.56s/trial, best loss: 0.0027436478587617797]

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a Num

 92%|█████████▏| 23/25 [04:11<00:22, 11.46s/trial, best loss: 0.0027436478587617797]

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a Num

 96%|█████████▌| 24/25 [04:22<00:11, 11.26s/trial, best loss: 0.0027436478587617797]

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  rval.shape = size

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\hyperopt\pyll\stochastic.py:144: DeprecationWarning: Setting the shape on a Num

100%|██████████| 25/25 [04:35<00:00, 11.04s/trial, best loss: 0.0010736013360372182]
best:
{'min_cluster_size': 10, 'n_components': 21, 'n_neighbors': 18, 'random_state': 42}
label count: 12
data_clustered.head()
                                                                                                                                                     text  \
0                                                                                                   game actually made want take video game art seriously   
1                                                                                                                          awesome game like previous two   
2                                                              might feel atmosphere first two game nice storyline fun vibrant city run around kill enemy   
3  expect much dlc played ba ep found motivation play second episode yet main story good enough make disappointment dlc season pas bought game first came   
4 